In [1]:
import pandas as pd
import numpy as np

# Configuración visual: le decimos a Pandas que nos muestre TODAS las columnas sin cortarlas
pd.set_option('display.max_columns', None)

print("Iniciando entorno de limpieza de datos...")

# Cargamos el archivo especificando el formato europeo que le dimos antes
df = pd.read_csv('dataset_unificado_bruto.csv', sep=';', decimal=',')

print(f" Dataset cargado con éxito. Tenemos {df.shape[0]} pisos y {df.shape[1]} columnas.")

# Imprimimos las 5 primeras filas para comprobar que todo está en su sitio
df.head()

Iniciando entorno de limpieza de datos...
 Dataset cargado con éxito. Tenemos 1995 pisos y 22 columnas.


C:\Users\DANI\AppData\Local\Temp\ipykernel_21312\3394785440.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,Enlace,Precio,Superficie_Construida,Superficie_Util,Habitaciones,Banos,Planta,Antiguedad,Referencia,Descripcion,Ascensor,Garaje,Trastero,Terraza,Balcon,Jardin,Piscina,Aire_Acondicionado,Estado,Exterior,Gastos_Comunidad,Origen
0,https://www.pisos.com/comprar/piso-russafa4600...,840000.0,140.0,128.0,3.0,2.0,2.0,Más de 50 años,13931/5458,Grupo Alain inmobiliaria vende este exclusivo ...,Sí,No,No,No,No,No,No,Sí,NaN,NaN,NaN,Pisos.com
1,https://www.pisos.com/comprar/piso-el_carme460...,319000.0,64.0,60.0,NaN,NaN,2.0,Más de 50 años,13913/5458,Grupo Alain inmobiliaria vende piso en la zona...,Sí,No,No,No,Sí,No,No,Sí,NaN,NaN,NaN,Pisos.com
2,https://www.pisos.com/comprar/piso-eixample_gr...,620000.0,108.0,100.0,4.0,2.0,5.0,NaN,14189/5458,Grupo Alain Inmobiliaria vede un impresionante...,Sí,No,No,No,Sí,No,No,Sí,NaN,NaN,NaN,Pisos.com
3,https://www.pisos.com/comprar/atico-russafa460...,650000.0,100.0,95.0,4.0,2.0,5.0,Más de 50 años,REF.H1163V,HELICE Inmobiliaria presenta este ático dúplex...,Sí,No,No,Sí,Sí,Sí,No,Sí,Reformado,Sí,NaN,Pisos.com
4,https://www.pisos.com/comprar/piso-eixample_gr...,625000.0,127.0,115.0,4.0,2.0,4.0,Más de 50 años,RM087,Vivienda en exclusiva en la zona del example v...,Sí,No,No,No,Sí,No,No,Sí,En buen estado,Sí,Más de 100 €,Pisos.com


Ahora limpiamos la columna de ascensor para que tenga en cuenta bien las descripciones y si pone sin ascensor que no rellene con un si porque sería un falso positivo y adulteraría el resultado final

In [2]:
import re

print("Iniciando auditoría semántica de la variable 'Ascensor'...")

# 1. Guardamos una copia de lo que dijo el scraper para comparar después
df['Ascensor_Original'] = df['Ascensor'].copy()

# 2. Preparamos el texto de la descripción (todo en minúsculas) para buscar bien
df['Texto_Busqueda'] = df['Descripcion'].fillna('').astype(str).str.lower()

# 3. La Función Detectora
def filtro_estricto_ascensor(row):
    texto = row['Texto_Busqueda']
    
    # Patrones de negación típicos de las inmobiliarias
    trampas = r'sin ascensor|no dispone de ascensor|no tiene ascensor|carece de ascensor|finca sin ascensor|proyecto de ascensor|no hay ascensor'
    
    # Si encuentra alguna de esas trampas en el texto, el ascensor es falso
    if re.search(trampas, texto):
        return 'No'
    
    # Si no hay trampa, respetamos lo que capturó el scraper originalmente
    return str(row['Ascensor_Original'])

# 4. Aplicamos el filtro fila por fila
df['Ascensor'] = df.apply(filtro_estricto_ascensor, axis=1)

# 5. Calculamos el impacto para tu TFM
falsos_positivos_corregidos = len(df[(df['Ascensor_Original'] == 'Sí') & (df['Ascensor'] == 'No')])

# 6. Borramos las columnas temporales que usamos para buscar
df = df.drop(columns=['Ascensor_Original', 'Texto_Busqueda'])

print("-" * 50)
print(f" ¡Limpieza completada con éxito!")
print(f" Se han detectado y corregido {falsos_positivos_corregidos} inmuebles que decían tener ascensor falsamente.")
print("-" * 50)

# Mostramos cómo ha quedado el recuento final real
print("\nDistribución real actual:")
print(df['Ascensor'].value_counts())

Iniciando auditoría semántica de la variable 'Ascensor'...
--------------------------------------------------
 ¡Limpieza completada con éxito!
 Se han detectado y corregido 172 inmuebles que decían tener ascensor falsamente.
--------------------------------------------------

Distribución real actual:
Ascensor
Sí    1377
No     618
Name: count, dtype: int64


In [3]:
import re

print("Iniciando auditoría semántica ULTRAPRECISA de la variable 'Garaje'...")

# 1. Guardamos copia original
df['Garaje_Original'] = df['Garaje'].copy()

# 2. Preparamos el texto (quitamos saltos de línea y pasamos a minúsculas)
df['Texto_Busqueda'] = df['Descripcion'].fillna('').astype(str).str.lower().str.replace('\n', ' ')

# 3. La Función Detectora (Red de Arrastre)
def filtro_dios_garaje(row):
    texto = row['Texto_Busqueda']
    
    # --- NIVEL 1: Negaciones directas y absolutas ---
    # Captura: "sin garaje", "no tiene/dispone/incluye garaje/plaza", "garaje/plaza no incluido/a"
    if re.search(r'sin garaje|no (tiene|dispone de|incluye) (garaje|plaza)|(garaje|plaza) no incluid[oa]', texto):
        return 'No'
        
    # --- NIVEL 2: Opcionalidad e Inversión Extra (Las palabras trampa) ---
    # Captura: "posibilidad de garaje", "invertir además en garaje", "adquirir plaza"
    if re.search(r'(posibilidad|opci[óo]n|oportunidad|adquirir|comprar|invertir).{0,40}(garaje|plaza)', texto):
        return 'No'
    # Captura: "garaje opcional", "plaza aparte", "garaje adicional"
    if re.search(r'(garaje|plaza).{0,25}(opcional|aparte|adicional|no incluid)', texto):
        return 'No'
        
    # --- NIVEL 3: Coste Extra (Búsqueda por proximidad matemática) ---
    # Detecta la palabra garaje/plaza, seguida de hasta 50 letras, y luego un número con "€" o "euros"
    # Ej: "Plaza de garaje + trastero por 35.000 €"
    if re.search(r'(garaje|plaza).{0,50}(por|desde|\+|:).{0,15}\d+.*(€|euros)', texto):
        return 'No'
    
    # Si sobrevive a este campo de minas, asumimos que es un garaje real
    return str(row['Garaje_Original'])

# 4. Aplicamos el filtro avanzado
df['Garaje'] = df.apply(filtro_dios_garaje, axis=1)

# 5. Calculamos el impacto destructivo
falsos_positivos_garaje = len(df[(df['Garaje_Original'] == 'Sí') & (df['Garaje'] == 'No')])

# 6. Borramos columnas temporales
df = df.drop(columns=['Garaje_Original', 'Texto_Busqueda'])

print("-" * 65)
print(f" ¡Limpieza de Garajes nivel Dios completada!")
print(f" Se han detectado y fulminado {falsos_positivos_garaje} garajes falsos o de pago extra.")
print("-" * 65)

# Mostramos la distribución final
print("\nDistribución real actual de Garajes:")
print(df['Garaje'].value_counts())

Iniciando auditoría semántica ULTRAPRECISA de la variable 'Garaje'...
-----------------------------------------------------------------
 ¡Limpieza de Garajes nivel Dios completada!
 Se han detectado y fulminado 118 garajes falsos o de pago extra.
-----------------------------------------------------------------

Distribución real actual de Garajes:
Garaje
No    1483
Sí     512
Name: count, dtype: int64


In [4]:
import re

print("Iniciando auditoría semántica de la variable 'Trastero'...")

# 1. Guardamos copia original
df['Trastero_Original'] = df['Trastero'].copy()

# 2. Preparamos el texto de búsqueda
df['Texto_Busqueda'] = df['Descripcion'].fillna('').astype(str).str.lower().str.replace('\n', ' ')

# 3. Función Detectora de Trasteros Falsos
def filtro_trastero_inteligente(row):
    texto = row['Texto_Busqueda']
    
    # --- NIVEL 1: Negaciones directas ---
    # Captura: "sin trastero", "no incluye trastero", "trastero no incluido"
    if re.search(r'sin trastero|no (tiene|dispone de|incluye) trastero|trastero no incluid[oa]', texto):
        return 'No'
        
    # --- NIVEL 2: Opcionalidad y Compra extra ---
    # Captura: "posibilidad de trastero", "trastero opcional", "opción a trastero"
    if re.search(r'(posibilidad|opci[óo]n|oportunidad|adquirir|comprar).{0,30}trastero', texto):
        return 'No'
    if re.search(r'trastero.{0,20}(opcional|aparte|adicional|no incluid)', texto):
        return 'No'
        
    # --- NIVEL 3: El factor precio (Proximidad) ---
    # Captura: "trastero por 5.000€", "trastero + 6000 euros"
    if re.search(r'trastero.{0,40}(por|desde|\+|:).{0,15}\d+.*(€|euros)', texto):
        return 'No'
    
    return str(row['Trastero_Original'])

# 4. Aplicamos el filtro
df['Trastero'] = df.apply(filtro_trastero_inteligente, axis=1)

# 5. Impacto en el Dataset
falsos_positivos_trastero = len(df[(df['Trastero_Original'] == 'Sí') & (df['Trastero'] == 'No')])

# 6. Limpieza de columnas temporales
df = df.drop(columns=['Trastero_Original', 'Texto_Busqueda'])

print("-" * 65)
print(f" ¡Limpieza de Trasteros completada!")
print(f" Se han detectado y corregido {falsos_positivos_trastero} trasteros que eran opcionales o de pago.")
print("-" * 65)

# Distribución final
print("\nDistribución real de Trasteros:")
print(df['Trastero'].value_counts())

Iniciando auditoría semántica de la variable 'Trastero'...
-----------------------------------------------------------------
 ¡Limpieza de Trasteros completada!
 Se han detectado y corregido 33 trasteros que eran opcionales o de pago.
-----------------------------------------------------------------

Distribución real de Trasteros:
Trastero
No    1585
Sí     410
Name: count, dtype: int64


In [5]:
import re

print("Iniciando auditoría semántica de 'Terraza' y 'Balcón'...")

# 1. Guardamos copias originales
df['Terraza_Original'] = df['Terraza'].copy()
df['Balcon_Original'] = df['Balcon'].copy()

# 2. Preparamos el texto
df['Texto_Busqueda'] = df['Descripcion'].fillna('').astype(str).str.lower().str.replace('\n', ' ')

# 3. Función Detectora para TERRAZAS
def filtro_terraza(row):
    texto = row['Texto_Busqueda']
    estado = str(row['Terraza_Original'])
    
    if estado == 'No': 
        return 'No'
        
    # Trampas: Terrazas comunitarias, compartidas, azoteas de la finca o negaciones puras
    trampas = r'sin terraza|no (tiene|dispone de) terraza|terraza (comunitaria|compartida)|azotea comunitaria|posibilidad de (hacer|sacar) terraza'
    
    if re.search(trampas, texto):
        return 'No'
        
    return estado

# 4. Función Detectora para BALCONES
def filtro_balcon(row):
    texto = row['Texto_Busqueda']
    estado = str(row['Balcon_Original'])
    
    if estado == 'No': 
        return 'No'
        
    # Trampas: Balcones franceses (no transitables), miradores o negaciones puras
    trampas = r'sin balc[oó]n|no (tiene|dispone de) balc[oó]n|balc[oó]n franc[eé]s|falso balc[oó]n'
    
    if re.search(trampas, texto):
        return 'No'
        
    return estado

# 5. Aplicamos los filtros
df['Terraza'] = df.apply(filtro_terraza, axis=1)
df['Balcon'] = df.apply(filtro_balcon, axis=1)

# 6. Calculamos los falsos positivos (Impacto)
fp_terraza = len(df[(df['Terraza_Original'] == 'Sí') & (df['Terraza'] == 'No')])
fp_balcon = len(df[(df['Balcon_Original'] == 'Sí') & (df['Balcon'] == 'No')])

# 7. Limpiamos las variables temporales
df = df.drop(columns=['Terraza_Original', 'Balcon_Original', 'Texto_Busqueda'])

print("-" * 65)
print(f" ¡Limpieza de Exteriores completada!")
print(f" Terrazas falsas o comunitarias eliminadas: {fp_terraza}")
print(f"🪴 Balcones franceses o falsos eliminados: {fp_balcon}")
print("-" * 65)

Iniciando auditoría semántica de 'Terraza' y 'Balcón'...
-----------------------------------------------------------------
 ¡Limpieza de Exteriores completada!
 Terrazas falsas o comunitarias eliminadas: 26
🪴 Balcones franceses o falsos eliminados: 3
-----------------------------------------------------------------


In [6]:
import re

print("Iniciando auditoría semántica de la variable 'Jardín' (Filtro Geográfico)...")

# 1. Guardamos copia original para el recuento
df['Jardin_Original'] = df['Jardin'].copy()

# 2. Preparamos el texto de búsqueda
df['Texto_Busqueda'] = df['Descripcion'].fillna('').astype(str).str.lower().str.replace('\n', ' ')

def corregir_falso_jardin(row):
    # Si el scraper ya dijo que no, nos ahorramos el proceso
    if str(row['Jardin_Original']).strip() == 'No':
        return 'No'

    texto = row['Texto_Busqueda']

    # --- PASO 1: SÍ ABSOLUTO (Jardín privado/comunitario real) ---
    # Si especifica que es privado, propio o comunitario, lo damos por bueno directamente
    if re.search(r'jard[ií]n (privado|propio|comunitario|exclusivo)|zonas comunes con jard[ií]n', texto):
        return 'Sí'

    # --- PASO 2: ELIMINACIÓN DE TRAMPAS (El Borrador Geográfico) ---
    # Listado de parques públicos de Valencia y frases de entorno
    trampas_valencia = [
        r'jard[ií]n(?:es)?\s+(?:del?|como\s+el\s+de|como)?\s*(?:turia|viveros|bot[aá]nico|real|ayora|monforte|cabecera)',
        r'antiguo cauce', r'viejo cauce', r'cauce del r[ií]o',
        r'(?:vistas a|junto a|rodeado de|cerca de|proximidad a|a pocos metros de).{0,30}(?:parques?|jard[ií]n(?:es)?|zonas verdes)',
        r'jardines\s+(?:del entorno|de la zona|de los alrededores|p[úu]blicos|municipales)',
        r'parques y jardines'
    ]
    
    texto_saneado = texto
    for patron in trampas_valencia:
        texto_saneado = re.sub(patron, ' [BORRADO] ', texto_saneado)

    # --- PASO 3: VERIFICACIÓN FINAL ---
    # Buscamos la palabra "jardín" o "jardines" en el texto donde ya no están los parques públicos
    # Usamos \b para que no detecte "jardinera" o "jardinería"
    if re.search(r'\bjard[ií]n\b|\bjardines\b|\bzonas verdes\b', texto_saneado):
        return "Sí"
        
    return "No"

# 3. Aplicamos la cirugía
df['Jardin'] = df.apply(corregir_falso_jardin, axis=1)

# 4. Calculamos el impacto para tu TFM
falsos_jardines = len(df[(df['Jardin_Original'] == 'Sí') & (df['Jardin'] == 'No')])

# 5. Limpiamos temporales
df = df.drop(columns=['Jardin_Original', 'Texto_Busqueda'])

print("-" * 65)
print(f" ¡Auditoría de Jardines finalizada!")
print(f" Se han detectado {falsos_jardines} inmuebles que 'alquilaban' el cauce del río como jardín propio.")
print("-" * 65)

# Distribución final
print(df['Jardin'].value_counts())

Iniciando auditoría semántica de la variable 'Jardín' (Filtro Geográfico)...
-----------------------------------------------------------------
 ¡Auditoría de Jardines finalizada!
 Se han detectado 384 inmuebles que 'alquilaban' el cauce del río como jardín propio.
-----------------------------------------------------------------
Jardin
No    1784
Sí     211
Name: count, dtype: int64


In [7]:
import re

print("Iniciando auditoría final: Piscina y Climatización...")

# 1. Guardamos copias originales
df['Piscina_Original'] = df['Piscina'].copy()
df['Aire_Original'] = df['Aire_Acondicionado'].copy()

# 2. Preparamos el texto
df['Texto_Busqueda'] = df['Descripcion'].fillna('').astype(str).str.lower().str.replace('\n', ' ')

# 3. Función Detectora para PISCINA
def filtro_piscina(row):
    texto = row['Texto_Busqueda']
    estado = str(row['Piscina_Original'])
    
    if estado == 'No': return 'No'
    
    # Trampas: Piscinas públicas, municipales, polideportivos o "vistas a la piscina" (del vecino)
    trampas = r'piscina municipal|polideportivo|piscina del club|piscina de la zona|piscina p[úu]blica|vistas a la piscina|cerca de la piscina'
    # También posibilidades futuras
    posibilidades = r'posibilidad de piscina|espacio para piscina|proyecto de piscina'
    
    if re.search(trampas, texto) or re.search(posibilidades, texto):
        return 'No'
    
    return estado

# 4. Función Detectora para AIRE ACONDICIONADO
def filtro_aire(row):
    texto = row['Texto_Busqueda']
    estado = str(row['Aire_Original'])
    
    if estado == 'No': return 'No'
    
    # La gran trampa: Preinstalación (no hay máquinas, solo el hueco)
    if re.search(r'preinstalaci[oó]n|pre-instalaci[oó]n', texto):
        # Si SOLO dice preinstalación y no menciona "split" o "conductos funcionando"
        if not re.search(r'splits? instalados|aire por conductos|bomba de fr[íi]o', texto):
            return 'No'
            
    # Otras trampas: "no funciona", "posibilidad de poner", "no incluye las máquinas"
    trampas_ac = r'aire acondicionado no funciona|posibilidad de instalar aire|no incluye aire|sin aire acondicionado'
    
    if re.search(trampas_ac, texto):
        return 'No'
        
    return estado

# 5. Aplicamos los filtros
df['Piscina'] = df.apply(filtro_piscina, axis=1)
df['Aire_Acondicionado'] = df.apply(filtro_aire, axis=1)

# 6. Impacto
fp_piscina = len(df[(df['Piscina_Original'] == 'Sí') & (df['Piscina'] == 'No')])
fp_aire = len(df[(df['Aire_Original'] == 'Sí') & (df['Aire_Acondicionado'] == 'No')])

# 7. Limpieza de temporales
df = df.drop(columns=['Piscina_Original', 'Aire_Original', 'Texto_Busqueda'])

print("-" * 65)
print(f"¡Auditoría final completada!")
print(f" Piscinas municipales o 'vistas a piscina' eliminadas: {fp_piscina}")
print(f" Preinstalaciones o máquinas que no funcionan eliminadas: {fp_aire}")
print("-" * 65)

Iniciando auditoría final: Piscina y Climatización...
-----------------------------------------------------------------
¡Auditoría final completada!
 Piscinas municipales o 'vistas a piscina' eliminadas: 21
 Preinstalaciones o máquinas que no funcionan eliminadas: 13
-----------------------------------------------------------------


In [8]:
import re

print("Re-ejecutando el Rescatador de Jardines (Versión 'Anti-Barras')...")

def rescatador_perfecto_jardin(row):
    enlace = str(row['Enlace']).lower().strip()
    
    # 1. Quitamos la barra final si existe y sacamos la última parte
    # Ejemplo: '.../casa-el_perellonet/' -> '.../casa-el_perellonet' -> 'casa-el_perellonet'
    slug = enlace.rstrip('/').split('/')[-1]
    
    # 2. Definimos los términos
    terminos_casa = ['casa', 'chalet', 'adosado', 'unifamiliar', 'villa', 'finca', 'torre']
    terminos_piso = ['piso', 'apartamento', 'duplex', 'atico', 'estudio', 'loft', 'planta-baja']
    
    # 3. LÓGICA DE DECISIÓN
    
    # REGLA ORO: Si en el slug aparece 'piso', 'apartamento', etc. -> NO tiene jardín.
    if any(tipo in slug for tipo in terminos_piso):
        return 'No'
    
    # REGLA PLATA: Si en el slug aparece 'casa', 'chalet', etc.
    if any(tipo in slug for tipo in terminos_casa):
        # Miramos si la descripción menciona jardín, patio o parcela
        descripcion = str(row['Descripcion']).lower()
        if re.search(r'\bjard[ií]n\b|\bjardines\b|parcela|terreno|patio|solar', descripcion):
            # Filtro de seguridad para el Río Turia
            if not re.search(r'vistas al jard[ií]n|junto al jard[ií]n del turia', descripcion):
                return 'Sí'
    
    return 'No'

# Aplicamos la limpieza
df['Jardin'] = df.apply(rescatador_perfecto_jardin, axis=1)

# 4. Recuento
jardines_encontrados = len(df[df['Jardin'] == 'Sí'])

print("-" * 65)
print(f" ¡Rescate completado con éxito!")
print(f" Se han encontrado {jardines_encontrados} inmuebles que son casas reales con jardín.")
print(f" Se han descartado todos los pisos de pisos.com.")
print("-" * 65)

if jardines_encontrados > 0:
    print("\nEjemplos de casas con jardín rescatadas:")
    print(df[df['Jardin'] == 'Sí'][['Enlace', 'Precio']].head(10))

Re-ejecutando el Rescatador de Jardines (Versión 'Anti-Barras')...
-----------------------------------------------------------------
 ¡Rescate completado con éxito!
 Se han encontrado 60 inmuebles que son casas reales con jardín.
 Se han descartado todos los pisos de pisos.com.
-----------------------------------------------------------------

Ejemplos de casas con jardín rescatadas:
                                                Enlace     Precio
17   https://www.pisos.com/comprar/casa-el_cabanyal...   315000.0
36   https://www.pisos.com/comprar/casa-el_cabanyal...   235000.0
104  https://www.pisos.com/comprar/casa-el_cabanyal...  1400000.0
154  https://www.pisos.com/comprar/casa_adosada-san...   830000.0
273  https://www.pisos.com/comprar/casa-el_castella...   250000.0
281  https://www.pisos.com/comprar/chalet-arrancapi...   690000.0
410  https://www.pisos.com/comprar/casa-pobles_del_...   950000.0
433  https://www.pisos.com/comprar/casa-el_cabanyal...  1500000.0
445  https://www.pi

In [9]:
print("Iniciando purga de activos rústicos, parcelas y solares...")

# 1. Guardamos el total actual para ver el impacto
total_antes_purga = len(df)

# 2. Definimos la 'Lista Negra' de términos que no queremos en nuestro modelo urbano
# Estos términos suelen indicar terrenos baldíos o casas de campo sin servicios urbanos
lista_negra_rustica = ['rustica', 'finca-rustica', 'parcela', 'solar', 'terreno', 'agricola', 'huerto']

# 3. Aplicamos el filtro: Solo nos quedamos con los que NO contengan estas palabras en la URL
# Usamos el slug (la parte final del enlace) que es donde está la categoría
df['slug_temp'] = df['Enlace'].str.lower().str.rstrip('/').str.split('/').str[-1]

# Filtramos: Buscamos filas donde NINGÚN término de la lista negra esté en el slug
df = df[~df['slug_temp'].str.contains('|'.join(lista_negra_rustica), na=False)].copy()

# 4. Limpieza de columna temporal y cálculo de impacto
df = df.drop(columns=['slug_temp'])
eliminados_rusticos = total_antes_purga - len(df)

print("-" * 65)
print(f" ¡Purga de activos no urbanos completada!")
print(f" Se han eliminado {eliminados_rusticos} registros (rústicos/terrenos/solares).")
print(f" Dataset final optimizado para entorno urbano: {len(df)} inmuebles.")
print("-" * 65)

Iniciando purga de activos rústicos, parcelas y solares...
-----------------------------------------------------------------
 ¡Purga de activos no urbanos completada!
 Se han eliminado 8 registros (rústicos/terrenos/solares).
 Dataset final optimizado para entorno urbano: 1987 inmuebles.
-----------------------------------------------------------------


In [10]:
print("Iniciando purga de activos de bajo coste y estudios sin habitaciones...")

# 1. Registro antes de la limpieza
total_antes_filtros = len(df)

# 2. Aplicamos el Filtro de Precio (Mínimo 100.000€)
df = df[df['Precio'] >= 100000].copy()
eliminados_por_precio = total_antes_filtros - len(df)

# 3. Aplicamos el Filtro de Habitaciones (Mínimo 1 habitación)
# Esto elimina estudios (0 habs) y posibles errores de datos
antes_habs = len(df)
df = df[df['Habitaciones'] > 0].copy()
eliminados_por_habs = antes_habs - len(df)

print("-" * 65)
print(f" Limpieza de coherencia comercial completada.")
print(f" Inmuebles eliminados por precio < 100.000€: {eliminados_por_precio}")
print(f" Inmuebles eliminados por tener 0 habitaciones: {eliminados_por_habs}")
print(f" Registros restantes en el dataset: {len(df)}")
print("-" * 65)


Iniciando purga de activos de bajo coste y estudios sin habitaciones...
-----------------------------------------------------------------
 Limpieza de coherencia comercial completada.
 Inmuebles eliminados por precio < 100.000€: 40
 Inmuebles eliminados por tener 0 habitaciones: 56
 Registros restantes en el dataset: 1891
-----------------------------------------------------------------


In [11]:
import numpy as np

print("Iniciando reconstrucción de la variable 'Planta'...")

def reconstruir_planta(row):
    planta_actual = row['Planta']
    descripcion = str(row['Descripcion']).lower()
    enlace = row['Enlace'].lower()
    
    # Si ya tenemos un número de planta válido y no es NaN, lo dejamos tal cual
    if pd.notna(planta_actual) and str(planta_actual).replace('.','').isdigit():
        return float(planta_actual)
    
    # --- CASO 1: ÁTICOS ---
    if 'atico' in descripcion or 'ático' in descripcion or 'atico' in enlace:
        # Les asignamos una planta alta simbólica (ej. 10) o la máxima del dataset
        # Para el TFM, lo ideal es marcarlo como una planta alta.
        return 8.0 
    
    # --- CASO 2: BAJOS / PLANTAS BAJAS ---
    if 'bajo' in descripcion or 'planta baja' in descripcion or 'piso-bajo' in enlace:
        return 0.0
    
    # --- CASO 3: ENTRESUELOS (Muy comunes en Valencia) ---
    if 'entresuelo' in descripcion or 'entresuelo' in enlace:
        return 0.5
    
    # Si no encontramos nada, devolvemos NaN para decidir luego
    return np.nan

# Aplicamos la reconstrucción
df['Planta'] = df.apply(reconstruir_planta, axis=1)

# 3. ¿Qué hacemos con los que siguen siendo vacíos?
# Contamos cuántos quedan
vacios_finales = df['Planta'].isna().sum()

print(f" Reconstrucción finalizada.")
print(f" Quedan {vacios_finales} filas sin planta definida.")

# DECISIÓN FINAL PARA EL TFM:
# Si después de buscar en el texto siguen siendo NaN, lo mejor es:
# Rellenar con la MEDIANA (la planta más común en Valencia suele ser un 2º o 3º)
mediana_planta = df['Planta'].median()
df['Planta'] = df['Planta'].fillna(mediana_planta)

print(f"Sustituidos los vacíos restantes por la mediana: {mediana_planta}")

Iniciando reconstrucción de la variable 'Planta'...
 Reconstrucción finalizada.
 Quedan 269 filas sin planta definida.
Sustituidos los vacíos restantes por la mediana: 3.0


In [12]:
import re

print("Clasificando tipologías por presencia LITERAL en la descripción...")

def clasificar_tipologia_literal(row):
    # Convertimos la descripción a minúsculas para la búsqueda
    texto = str(row['Descripcion']).lower()
    
    # 1. Búsqueda literal de DÚPLEX
    # El \b asegura que sea la palabra exacta y no parte de otra
    if re.search(r'\bd[uú]plex\b', texto):
        return 'Dúplex'
    
    # 2. Búsqueda literal de ÁTICO
    if re.search(r'\b[aá]tico\b', texto):
        return 'Ático'
    
    # 3. Si no aparece ninguna de las anteriores, es Estandard
    return 'Estandard'

# Aplicamos la función a la columna 'Descripcion'
df['Tipologia_Especial'] = df.apply(clasificar_tipologia_literal, axis=1)

# Recuento final de veracidad
conteo_literal = df['Tipologia_Especial'].value_counts()

print("-" * 50)
print(" Clasificación literal completada.")
print(conteo_literal)
print("-" * 50)


Clasificando tipologías por presencia LITERAL en la descripción...
--------------------------------------------------
 Clasificación literal completada.
Tipologia_Especial
Estandard    1721
Dúplex         87
Ático          83
Name: count, dtype: int64
--------------------------------------------------


In [13]:
import re

print("Rescatando el valor del Jardín para las 95 casas detectadas...")

def rescate_final_casas(row):
    enlace = str(row['Enlace']).lower()
    descripcion = str(row['Descripcion']).lower()
    
    # 1. Definimos qué es una CASA REAL por URL
    es_casa = any(tipo in enlace for tipo in ['casa-', 'chalet-', 'adosado-', 'unifamiliar-'])
    
    # 2. Si es una casa, buscamos si tiene jardín/patio/parcela
    if es_casa:
        if re.search(r'\bjard[ií]n\b|parcela|patio|terreno|solar', descripcion):
            # Excluimos si solo habla de las vistas al jardín del Turia
            if "vistas al" not in descripcion[:100]: 
                return 'Sí'
    
    # 3. Para todo lo que sea 'piso', 'ático' o 'duplex' -> Jardín = No
    return 'No'

# Aplicamos el filtro quirúrgico
df['Jardin'] = df.apply(rescate_final_casas, axis=1)

# Recuento de justicia
casas_con_jardin = len(df[df['Jardin'] == 'Sí'])

print("-" * 65)
print(f" ¡Proceso completado!")
print(f" Hemos identificado {casas_con_jardin} casas con espacio exterior real.")
print(f" Los ~1.800 pisos ahora tienen 'No', eliminando el ruido para la IA.")
print("-" * 65)

# Verificamos un par de ejemplos para tu TFM
print(df[df['Jardin'] == 'Sí'][['Enlace', 'Precio']].head(3))

Rescatando el valor del Jardín para las 95 casas detectadas...
-----------------------------------------------------------------
 ¡Proceso completado!
 Hemos identificado 39 casas con espacio exterior real.
 Los ~1.800 pisos ahora tienen 'No', eliminando el ruido para la IA.
-----------------------------------------------------------------
                                                Enlace     Precio
17   https://www.pisos.com/comprar/casa-el_cabanyal...   315000.0
104  https://www.pisos.com/comprar/casa-el_cabanyal...  1400000.0
273  https://www.pisos.com/comprar/casa-el_castella...   250000.0


In [14]:
print("Normalizando categorías de la columna 'Estado'...")

def normalizar_estado(valor):
    # Convertimos a minúsculas y limpiamos espacios para comparar mejor
    val = str(valor).lower().strip()
    
    # 1. Categoría: A REFORMAR
    if any(keyword in val for keyword in ['a reformar', 'para reformar', 'estado de origen']):
        return 'A reformar'
    
    # 2. Categoría: A ESTRENAR (Obra Nueva)
    if any(keyword in val for keyword in ['a estrenar', 'obra nueva', 'promoción']):
        return 'A estrenar'
    
    # 3. Categoría: REFORMADO
    if any(keyword in val for keyword in ['reformado', 'remodelado', 'semireformado']):
        # Evitamos que "a reformar" caiga aquí por contener la palabra "reformar"
        if 'a reformar' in val or 'para reformar' in val:
            return 'A reformar'
        return 'Reformado'
    
    # 4. Categoría: EN BUEN ESTADO (Segunda mano)
    if any(keyword in val for keyword in ['buen estado', 'segunda mano', 'conservado', 'habitables']):
        return 'En buen estado'
    
    # Si no encaja en nada o está vacío, le asignamos la más común por defecto (En buen estado)
    # o 'Desconocido' si prefieres ser más estricto.
    return 'En buen estado'

# Aplicamos la normalización
df['Estado'] = df['Estado'].apply(normalizar_estado)

# Calculamos el recuento para ver cómo ha quedado el mapa
conteo_final = df['Estado'].value_counts()

print("-" * 65)
print("¡Estandarización completada!")
print(f" Distribución final de estados:")
print(conteo_final)
print("-" * 65)

# Guardamos la versión definitiva del dataset con todo limpio
df.to_csv('dataset_VALENCIA_FINAL_ML.csv', index=False, sep=';', decimal=',')
print("💾 ¡Dataset maestro 'dataset_VALENCIA_FINAL_ML.csv' generado con éxito!")

Normalizando categorías de la columna 'Estado'...
-----------------------------------------------------------------
¡Estandarización completada!
 Distribución final de estados:
Estado
En buen estado    1635
Reformado          111
A reformar          89
A estrenar          56
Name: count, dtype: int64
-----------------------------------------------------------------
💾 ¡Dataset maestro 'dataset_VALENCIA_FINAL_ML.csv' generado con éxito!
